# Gemma 2 9B 파인튜닝 노트북
**대상 GPU**: Google Colab T4 (16GB VRAM)  
**기법**: QLoRA (4-bit) + SFT via Unsloth

## 실행 전 체크리스트
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. Google Drive에 프로젝트 폴더 업로드 (`gymbarofit/gemma2/`)
3. 셀을 순서대로 실행

## 셀 1: GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 유형을 T4 GPU로 변경하세요.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "T4" not in gpu_name:
    print("[WARNING] T4가 아닙니다. 메모리 설정을 재확인하세요.")

## 셀 2: 패키지 설치 (uv)

In [ ]:
# uv 설치 후 pyproject.toml 의존성 일괄 설치
!pip install uv -q

# pyproject.toml 기반 설치 (--system: Colab 기본 환경에 직접 설치)
!uv pip install --system -e .

print("설치 완료")

## 셀 3: W&B 로그인

In [ ]:
import wandb

WANDB_API_KEY = ""  # https://wandb.ai/settings 에서 복사

wandb.login(key=WANDB_API_KEY)

## 셀 4: Google Drive 마운트 + 프로젝트 경로 설정

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# Drive 내 프로젝트 경로 (본인 경로에 맞게 수정)
PROJECT_DIR = "/content/drive/MyDrive/gymbarofit/gemma2"
os.chdir(PROJECT_DIR)

# 필수 파일 존재 확인
required = [
    "data/train.jsonl",
    "data/val.jsonl",
    "data/dataset_config.json",
    "train.py",
    "config/lora_config.yaml",
    "pyproject.toml",
]
for path in required:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {path}")

## 셀 5: 데이터 샘플 확인

In [ ]:
import json

with open("data/dataset_config.json", encoding="utf-8") as f:
    config = json.load(f)
print("[Instruction]")
print(config["instruction"])
print()

with open("data/train.jsonl", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("[Sample Input]")
print(sample["input"])
print()
print("[Sample Output]")
print(sample["output"])

## 셀 6: 학습 실행

778건 × 3 epoch 기준 T4에서 약 **30~50분** 소요됩니다.

In [ ]:
!python train.py

## 셀 7: Loss 시각화

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# trainer_state.json 에서 로그 읽기
state_path = Path("models/gemma2-fitness-lora/trainer_state.json")
if not state_path.exists():
    # 체크포인트 디렉터리에서 탐색
    for p in Path("models/gemma2-fitness-lora").glob("checkpoint-*/trainer_state.json"):
        state_path = p
        break

with open(state_path, encoding="utf-8") as f:
    state = json.load(f)

logs = state["log_history"]
train_steps  = [l["step"] for l in logs if "loss" in l]
train_losses = [l["loss"] for l in logs if "loss" in l]
eval_steps   = [l["step"] for l in logs if "eval_loss" in l]
eval_losses  = [l["eval_loss"] for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_losses, label="Train Loss", alpha=0.8)
plt.plot(eval_steps, eval_losses, label="Val Loss", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Gemma 2 9B SFT — Loss Curve")
plt.legend()
plt.tight_layout()
plt.savefig("models/loss_curve.png", dpi=150)
plt.show()
print(f"최종 train loss: {train_losses[-1]:.4f}")
print(f"최종 val   loss: {eval_losses[-1]:.4f}")

## 셀 8: 추론 테스트

In [ ]:
from unsloth import FastLanguageModel
import torch, json

ADAPTER_PATH = "models/gemma2-fitness-lora"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# 테스트 입력
with open("data/dataset_config.json", encoding="utf-8") as f:
    instruction = json.load(f)["instruction"]

test_input = (
    "나이: 28세 | 성별: 여성 | 체중: 58.0kg | 키: 1.65m | BMI: 21.3(정상)\n"
    "체지방률: 24.0% | 안정시 심박수: 62bpm\n"
    "운동 유형: 근력 | 경험 수준: 초급 | 주 3회 운동\n"
    "이번 세션: 1.0시간 | 평균 심박수: 135bpm | 최대 심박수: 175bpm\n"
    "소모 칼로리: 420kcal | 수분 섭취: 1.5L"
)

prompt = (
    f"<start_of_turn>user\n{instruction}\n\n{test_input}<end_of_turn>\n"
    f"<start_of_turn>model\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.1,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

## 셀 9: 어댑터 Drive 백업

In [ ]:
import shutil, os

BACKUP_DIR = "/content/drive/MyDrive/gymbarofit/gemma2/models/gemma2-fitness-lora"
os.makedirs(BACKUP_DIR, exist_ok=True)

# adapter_model.safetensors + adapter_config.json + tokenizer 파일 복사
src = "models/gemma2-fitness-lora"
for fname in os.listdir(src):
    if fname.endswith(("safetensors", ".json", ".model")):
        shutil.copy(os.path.join(src, fname), BACKUP_DIR)
        print(f"  복사: {fname}")

print("Drive 백업 완료")